# vad_card

> VAD chunk card renderer for the alignment card stack

In [ ]:
#| default_exp components.vad_card

In [ ]:
#| export
from typing import Any, Callable, Optional

from fasthtml.common import Div, Span

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles
from cjm_fasthtml_daisyui.components.data_display.card import card, card_body
from cjm_fasthtml_daisyui.components.feedback.loading import loading, loading_styles, loading_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, font_family
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, items, justify, gap, grow, shrink
)
from cjm_fasthtml_tailwind.utilities.layout import position, right, top, visibility
from cjm_fasthtml_tailwind.utilities.transforms import translate
from cjm_fasthtml_tailwind.utilities.effects import opacity
from cjm_fasthtml_tailwind.utilities.transitions_and_animation import transition, duration
from cjm_fasthtml_tailwind.core.base import combine_classes

# Card stack library
from cjm_fasthtml_card_stack.core.constants import CardRole
from cjm_fasthtml_card_stack.core.models import CardRenderContext

# Local imports
from cjm_transcript_vad_align.html_ids import AlignmentHtmlIds
from cjm_transcript_vad_align.models import VADChunk
from cjm_transcript_vad_align.utils import format_time_precise

## Card Renderer

Each VAD chunk card is a compact row showing time range and duration.
Much simpler than segment cards — no split mode, no token selector.

In [ ]:
#| export
def render_vad_card(
    chunk:VADChunk,  # VAD chunk to render
    card_role:CardRole,  # Role of this card in viewport ("focused" or "context")
) -> Any:  # VAD chunk card component
    """Render a single VAD chunk card with time range, duration, and playing indicator."""
    is_focused = card_role == "focused"

    # Time range display
    time_range = Span(
        f"{format_time_precise(chunk.start_time)} \u2192 {format_time_precise(chunk.end_time)}",
        cls=combine_classes(
            font_size.sm, font_family.mono,
            text_dui.base_content if is_focused else text_dui.base_content.opacity(70)
        )
    )

    # Duration badge
    duration_badge = Span(
        f"{chunk.duration:.1f}s",
        cls=combine_classes(badge, badge_styles.ghost, font_size.xs, font_family.mono)
    )

    # Playing indicator — hidden by default, toggled visible by JS during audio playback
    playing_indicator = Div(
        Span(cls=combine_classes(loading, loading_styles.bars, loading_sizes.xs, text_dui.secondary)),
        cls=combine_classes(
            "vad-playing-indicator",
            position.absolute, right(2), top("1/2"), translate.y("1/2").negative,
            visibility.invisible,
        )
    )

    return Div(
        Div(
            # Index badge
            Span(
                f"#{chunk.index + 1}",
                cls=combine_classes(
                    font_size.xs, font_family.mono, font_weight.bold,
                    w(10), shrink(0), opacity(50)
                )
            ),

            # Time range
            time_range,

            # Spacer
            Div(cls=str(grow())),

            # Duration badge
            duration_badge,

            cls=combine_classes(
                card_body, p(3),
                flex_display, items.center, gap(3)
            )
        ),

        # Absolutely positioned playing indicator
        playing_indicator,

        id=AlignmentHtmlIds.vad_chunk(chunk.index),
        cls=combine_classes(
            card, "vad-card",
            position.relative,
            bg_dui.base_100,
            w.full,
            transition.all, duration(150)
        ),
        data_chunk_index=str(chunk.index),
        data_start_time=str(chunk.start_time),
        data_end_time=str(chunk.end_time),
        data_card_role=card_role
    )


## Card Renderer Factory

Creates a callback compatible with the card stack library's `render_card` parameter.

In [ ]:
#| export
def create_vad_card_renderer() -> Callable:  # Card renderer callback: (item, CardRenderContext) -> FT
    """Create a card renderer callback for VAD chunk cards."""
    def _render(
        item:Any,  # VADChunk instance
        context:CardRenderContext,  # Render context from card stack library
    ) -> Any:  # Rendered VAD chunk card component
        """Render a VAD chunk card for the given item and viewport context."""
        return render_vad_card(
            chunk=item,
            card_role=context.card_role,
        )
    return _render


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()